# 14 - Do the facts help the reader?

Each account of the development sample was read by each model alone (K1) and with the facts block (K1_facts), with no ML output in either. Does the facts block help the reader tell laundering accounts from clean ones?

Plan fixed on 8 November 2026, before any answer was read: development sample only; positives and clean accounts counted apart; paired tests on the same accounts (exact McNemar; AUC-ROC change with a stratified paired bootstrap, 2,000 draws). The facts helped if the 95% interval of the AUC change is above 0; if it contains 0, a help was not shown.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import binomtest, norm
from sklearn.metrics import roc_auc_score

from skeptical_second_reader.paths import DATA_INTERIM
from skeptical_second_reader.pipeline import READ_ALONE, READ_FACTS
from skeptical_second_reader.reading_sample import (
    MATCHED_NAME,
    OUTSIDE_NAME,
    RANDOM_NAME,
)
from skeptical_second_reader.second_reader import NUM_CTX, SUSPICIOUS

MODELS = ["gemma4:latest", "qwen3.5:9b-q4_K_M"]
CONDITIONS = [READ_ALONE, READ_FACTS]
CLEAN = [MATCHED_NAME, RANDOM_NAME]
DRAWS = 2000
SEED = 14

dev = pd.read_parquet(DATA_INTERIM / "reading_dev_cases.parquet")["stratum"]
results = pd.read_parquet(DATA_INTERIM / "reading" / "results_dev.parquet")
ceilings = pd.read_parquet(DATA_INTERIM / "ceiling_facts_scores.parquet")
truth = (~dev.isin(CLEAN)).astype(int)
patterns = sorted(s for s in dev.unique() if s.startswith("pattern "))
ORDER = patterns + [OUTSIDE_NAME] + CLEAN
print(f"accounts {len(dev)}  positives {truth.sum()}  clean {(truth == 0).sum()}")

accounts 255  positives 135  clean 120


## 1. Integrity

Checks that every account was read once per condition with one model version, how many answers are not usable, and that no prompt was cut by the 4,096-token window (the facts block makes prompts longer).

In [2]:
assert set(results["model"]) == set(MODELS)
assert set(results["condition"]) == set(CONDITIONS)
assert not results.duplicated(["model", "condition", "account"]).any()
for (model, condition), part in results.groupby(["model", "condition"]):
    assert set(part["account"]) == set(dev.index), (model, condition)
assert (results["group"] == dev.loc[results["account"]].to_numpy()).all()
assert (results.groupby("model")["digest"].nunique() == 1).all()
assert results["think"].nunique() == 1

usable = results["decision"].notna() & results["confidence"].notna()
results["usable"] = usable
tokens = results.groupby(["model", "condition"]).agg(
    answers=("account", "size"),
    not_usable=("usable", lambda s: int((~s).sum())),
    max_prompt_tokens=("prompt_tokens", "max"),
    max_prompt_and_answer=(
        "prompt_tokens",
        lambda s: int((s + results.loc[s.index, "answer_tokens"]).max()),
    ),
    at_window=("prompt_tokens", lambda s: int((s >= NUM_CTX).sum())),
)
print(f"context window {NUM_CTX}")
print(tokens.to_string())

context window 4096
                             answers  not_usable  max_prompt_tokens  max_prompt_and_answer  at_window
model             condition                                                                          
gemma4:latest     K1             255           0               1546                   1784          0
                  K1_facts       255           0               1744                   2042          0
qwen3.5:9b-q4_K_M K1             255           0               1570                   1743          0
                  K1_facts       255           0               1768                   1942          0


## 2. Share flagged per stratum

How often each reader says suspicious in each group, with Wilson 95% intervals: recall on the positives, false alarms on the clean accounts (15 accounts per stratum, clean 60).

In [3]:
def wilson(k, n, level=0.95):
    if n == 0:
        return np.nan, np.nan
    z = norm.ppf(0.5 + level / 2)
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half


keep = results.groupby("account")["usable"].all()
paired_accounts = keep[keep].index
use = results[results["account"].isin(paired_accounts)].copy()
use["says"] = use["decision"] == SUSPICIOUS
use["score"] = np.where(use["says"], use["confidence"], 1 - use["confidence"])
print(f"accounts usable in every model and condition: {len(paired_accounts)}")

rows = []
for (model, condition, stratum), part in use.groupby(["model", "condition", "group"]):
    k, n = int(part["says"].sum()), len(part)
    low, high = wilson(k, n)
    rows.append([model, condition, stratum, k, n, k / n, low, high])
flagged = pd.DataFrame(
    rows,
    columns=["model", "condition", "stratum", "flagged", "n", "share", "low", "high"],
)
for model in MODELS:
    mine = flagged[flagged["model"] == model].copy()
    mine["text"] = mine.apply(
        lambda r: (
            f"{r['flagged']:>2}/{r['n']:<2} {r['share']:.2f} "
            f"[{r['low']:.2f}, {r['high']:.2f}]"
        ),
        axis=1,
    )
    print(model)
    print(
        mine.pivot(index="stratum", columns="condition", values="text")
        .reindex(ORDER)
        .to_string()
    )
    print()

accounts usable in every model and condition: 255
gemma4:latest
condition                                        K1                 K1_facts
stratum                                                                     
pattern BIPARTITE            9/15 0.60 [0.36, 0.80]   7/15 0.47 [0.25, 0.70]
pattern CYCLE               11/15 0.73 [0.48, 0.89]  11/15 0.73 [0.48, 0.89]
pattern FAN-IN              15/15 1.00 [0.80, 1.00]  13/15 0.87 [0.62, 0.96]
pattern FAN-OUT              4/15 0.27 [0.11, 0.52]   4/15 0.27 [0.11, 0.52]
pattern GATHER-SCATTER       6/15 0.40 [0.20, 0.64]   5/15 0.33 [0.15, 0.58]
pattern RANDOM               7/15 0.47 [0.25, 0.70]   8/15 0.53 [0.30, 0.75]
pattern SCATTER-GATHER      11/15 0.73 [0.48, 0.89]  12/15 0.80 [0.55, 0.93]
pattern STACK                9/15 0.60 [0.36, 0.80]   8/15 0.53 [0.30, 0.75]
positive, outside the file   9/15 0.60 [0.36, 0.80]   9/15 0.60 [0.36, 0.80]
clean, matched activity     30/60 0.50 [0.38, 0.62]  25/60 0.42 [0.30, 0.54]
clean, rando

## 3. Paired comparison: K1 against K1_facts

The main result: how many decisions become right or wrong with the facts (exact McNemar, positives and clean apart), and the AUC-ROC change on the same accounts (score = confidence when the reader says suspicious, one minus it otherwise).

In [4]:
rng = np.random.default_rng(SEED)
strata_index = [np.flatnonzero(dev.loc[paired_accounts].to_numpy() == s) for s in ORDER]
draws = [
    np.concatenate([rng.choice(ix, size=len(ix), replace=True) for ix in strata_index])
    for _ in range(DRAWS)
]
y = truth.loc[paired_accounts].to_numpy()

mcnemar_rows, auc_rows = [], []
for model in MODELS:
    wide = (
        use[use["model"] == model]
        .pivot(index="account", columns="condition", values=["says", "score"])
        .loc[paired_accounts]
    )
    right = {
        c: wide[("says", c)].astype(bool).to_numpy() == (y == 1) for c in CONDITIONS
    }
    for side, mask in [("positives", y == 1), ("clean", y == 0)]:
        better = int((~right[READ_ALONE] & right[READ_FACTS] & mask).sum())
        worse = int((right[READ_ALONE] & ~right[READ_FACTS] & mask).sum())
        p = binomtest(better, better + worse, 0.5).pvalue if better + worse else np.nan
        mcnemar_rows.append(
            [
                model,
                side,
                int(mask.sum()),
                int((right[READ_ALONE] & mask).sum()),
                int((right[READ_FACTS] & mask).sum()),
                better,
                worse,
                p,
            ]
        )
    alone = wide[("score", READ_ALONE)].to_numpy(float)
    facts = wide[("score", READ_FACTS)].to_numpy(float)
    a, f = roc_auc_score(y, alone), roc_auc_score(y, facts)
    boot = np.array(
        [roc_auc_score(y[d], facts[d]) - roc_auc_score(y[d], alone[d]) for d in draws]
    )
    low, high = np.percentile(boot, [2.5, 97.5])
    verdict = (
        "the facts helped"
        if low > 0
        else ("the facts hurt" if high < 0 else "a help was not shown")
    )
    auc_rows.append([model, a, f, f - a, low, high, verdict])

mcnemar = pd.DataFrame(
    mcnemar_rows,
    columns=[
        "model",
        "accounts",
        "n",
        "right K1",
        "right K1_facts",
        "became right",
        "became wrong",
        "exact p",
    ],
)
auc = pd.DataFrame(
    auc_rows,
    columns=[
        "model",
        "AUC K1",
        "AUC K1_facts",
        "change",
        "low 95%",
        "high 95%",
        "rule",
    ],
)
print("McNemar, decisions that change")
print(mcnemar.round(4).to_string(index=False))
print()
print(f"AUC-ROC, {len(paired_accounts)} accounts, stratified paired bootstrap {DRAWS}")
print(auc.round(3).to_string(index=False))

McNemar, decisions that change
            model  accounts   n  right K1  right K1_facts  became right  became wrong  exact p
    gemma4:latest positives 135        81              77             9            13   0.5235
    gemma4:latest     clean 120        68              76            10             2   0.0386
qwen3.5:9b-q4_K_M positives 135        99              87             0            12   0.0005
qwen3.5:9b-q4_K_M     clean 120        45              57            12             0   0.0005

AUC-ROC, 255 accounts, stratified paired bootstrap 2000
            model  AUC K1  AUC K1_facts  change  low 95%  high 95%                 rule
    gemma4:latest   0.597         0.610   0.012   -0.029     0.053 a help was not shown
qwen3.5:9b-q4_K_M   0.536         0.545   0.009   -0.032     0.051 a help was not shown


Change in the share flagged per stratum (K1_facts minus K1). Descriptive only: 15 accounts per stratum.

In [5]:
change = flagged.pivot_table(
    index="stratum", columns=["model", "condition"], values="share"
)
diff = pd.DataFrame(
    {
        model: change[(model, READ_FACTS)] - change[(model, READ_ALONE)]
        for model in MODELS
    }
).reindex(ORDER)
print(diff.round(3).to_string())

                            gemma4:latest  qwen3.5:9b-q4_K_M
stratum                                                     
pattern BIPARTITE                  -0.133              0.000
pattern CYCLE                       0.000             -0.067
pattern FAN-IN                     -0.133             -0.133
pattern FAN-OUT                     0.000              0.000
pattern GATHER-SCATTER             -0.067             -0.200
pattern RANDOM                      0.067             -0.133
pattern SCATTER-GATHER              0.067             -0.067
pattern STACK                      -0.067             -0.067
positive, outside the file          0.000             -0.133
clean, matched activity            -0.083             -0.083
clean, random                      -0.050             -0.117


## 4. Beside the ceiling

The classifier of notebooks 12 and 13 on the same evidence (prompt evidence for K1; evidence and facts for K1_facts), to show what the same information gives a trained model.

In [6]:
ceiling_names = {READ_ALONE: "prompt evidence", READ_FACTS: "evidence and facts"}
side = {}
for condition, name in ceiling_names.items():
    flags = ceilings.loc[paired_accounts, f"flags {name}"] == 1
    side[f"ceiling {condition}"] = flags.groupby(dev.loc[paired_accounts]).mean()
for model in MODELS:
    for condition in CONDITIONS:
        part = use[(use["model"] == model) & (use["condition"] == condition)]
        name = f"{model.split(':')[0]} {condition}"
        side[name] = part.groupby("group")["says"].mean()
print("share flagged per stratum")
print(pd.DataFrame(side).reindex(ORDER).round(2).to_string())
print()
aucs = {
    f"ceiling {c}": roc_auc_score(y, ceilings.loc[paired_accounts, f"scores {n}"])
    for c, n in ceiling_names.items()
}
for row in auc_rows:
    aucs[f"{row[0].split(':')[0]} {READ_ALONE}"] = row[1]
    aucs[f"{row[0].split(':')[0]} {READ_FACTS}"] = row[2]
print("AUC-ROC on the same accounts")
print(pd.Series(aucs).round(3).to_string())

share flagged per stratum
                            ceiling K1  ceiling K1_facts  gemma4 K1  gemma4 K1_facts  qwen3.5 K1  qwen3.5 K1_facts
pattern BIPARTITE                 0.27              0.27       0.60             0.47        0.73              0.73
pattern CYCLE                     0.60              0.53       0.73             0.73        0.93              0.87
pattern FAN-IN                    0.80              0.80       1.00             0.87        0.93              0.80
pattern FAN-OUT                   0.20              0.40       0.27             0.27        0.33              0.33
pattern GATHER-SCATTER            0.13              0.20       0.40             0.33        0.67              0.47
pattern RANDOM                    0.33              0.33       0.47             0.53        0.73              0.60
pattern SCATTER-GATHER            0.67              0.67       0.73             0.80        0.87              0.80
pattern STACK                     0.27              0.

## 5. Results

- AUC-ROC change: Gemma +0.012 [-0.029; +0.053], Qwen +0.009 [-0.032; +0.051]; a help was not shown for either model.
- Qwen moved one way, to `not suspicious`: on positives 0 decisions became right and 12 wrong, on clean accounts 12 right and 0 wrong (p = 0.0005 each). Gemma: positives 9 right, 13 wrong; clean 10 right, 2 wrong.
- Share flagged fell in most strata: recall in 4 (Gemma) and 7 (Qwen) of the 9 positive strata, false alarms in both clean strata for both models. The facts seem to move the threshold, not the ranking (an inference).
- The ceiling rose from 0.857 to 0.876 with the facts (no interval computed).